In [0]:
Primary_keys={
    "customers":"customer_id",
    "accounts":"account_id",
    "merchants":"merchant_id",
    "transactions":"transaction_id",
    "transaction_events":"event_id",
    "exchange_rates":"exchange_rate_id"
}
for table_name,primary_key in Primary_keys.items():
    df=spark.table(f"finstream_data_pipeline.bronze.{table_name}")
    total_count=df.count()
    distinct_primary_key=df.select(primary_key).distinct().count()
    duplicates=total_count-distinct_primary_key
    print(f"{table_name} has {total_count} records and {distinct_primary_key} distinct primary_key and {duplicates} duplicates")
    
   


customers has 10000 records and 10000 distinct primary_key and 0 duplicates
accounts has 15000 records and 15000 distinct primary_key and 0 duplicates
merchants has 2000 records and 2000 distinct primary_key and 0 duplicates
transactions has 100000 records and 100000 distinct primary_key and 0 duplicates
transaction_events has 291967 records and 291967 distinct primary_key and 0 duplicates
exchange_rates has 10950 records and 10950 distinct primary_key and 0 duplicates


In [0]:
TABLES = [
    "customers",
    "accounts",
    "merchants",
    "exchange_rates",
    "transactions",
    "transaction_events"
]
for table_name in TABLES:
    df = spark.table(f"finstream_data_pipeline.bronze.{table_name}")
    df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- country: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp_ntz (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- account_status: string (nullable = true)
 |-- opened_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

root
 |-- merchant_id: s

In [0]:
TABLES = [
    "customers",
    "accounts",
    "merchants",
    "exchange_rates",
    "transactions",
    "transaction_events"
]
from pyspark.sql.functions import col, sum as spark_sum

for table in TABLES:

    df = spark.table(
        f"finstream_data_pipeline.bronze.{table}"
    )

    print("\n" + "=" * 60)
    print(table.upper())
    print("=" * 60)

    null_counts = df.select([
        spark_sum(
            col(c).isNull().cast("int")
        ).alias(c)
        for c in df.columns
    ])

    null_counts.show()


CUSTOMERS
+-----------+----------+---------+-----+-----+-------------+-------+----------+----------+---------------+--------------------+------------+
|customer_id|first_name|last_name|email|phone|date_of_birth|country|created_at|updated_at|customer_status|_ingestion_timestamp|_source_file|
+-----------+----------+---------+-----+-----+-------------+-------+----------+----------+---------------+--------------------+------------+
|          0|         0|        0|    0|    0|            0|      0|         0|         0|              0|                   0|           0|
+-----------+----------+---------+-----+-----+-------------+-------+----------+----------+---------------+--------------------+------------+


ACCOUNTS
+----------+-----------+------------+--------+-------+--------------+---------+----------+--------------------+------------+
|account_id|customer_id|account_type|currency|balance|account_status|opened_at|updated_at|_ingestion_timestamp|_source_file|
+----------+-----------

In [0]:
transactions_df = spark.table(
    "finstream_data_pipeline.bronze.transactions"
)

transactions_df.groupBy(
    "transaction_type"
).agg(
    spark_sum(
        col("merchant_id").isNull().cast("int")
    ).alias("null_merchant_ids")
).show()

+----------------+-----------------+
|transaction_type|null_merchant_ids|
+----------------+-----------------+
|         payment|                0|
|        transfer|            20202|
|      withdrawal|            19862|
|        purchase|                0|
|         deposit|            19949|
+----------------+-----------------+



In [0]:
events_df = spark.table(
    "finstream_data_pipeline.bronze.transaction_events"
)
events_df.groupBy(
    "event_type"
).agg(
    spark_sum(
        col("failure_reason").isNull().cast("int")
    ).alias("null_failure_reasons")
).show()

+----------+--------------------+
|event_type|null_failure_reasons|
+----------+--------------------+
| initiated|              100000|
|authorized|               94948|
| completed|               89924|
|  reversed|                2043|
|    failed|                   0|
+----------+--------------------+



In [0]:
accounts_df = spark.table(
    "finstream_data_pipeline.bronze.accounts"
)

customers_df = spark.table(
    "finstream_data_pipeline.bronze.customers"
)

invalid_accounts = (
    accounts_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Accounts with invalid customer_id:",
    invalid_accounts.count()
)

Accounts with invalid customer_id: 0


In [0]:
transactions_df = spark.table(
    "finstream_data_pipeline.bronze.transactions"
)

invalid_transactions_accounts = (
    transactions_df
    .join(
        accounts_df.select("account_id"),
        on="account_id",
        how="left_anti"
    )
)

print(
    "Transactions with invalid account_id:",
    invalid_transactions_accounts.count()
)

Transactions with invalid account_id: 0


In [0]:
merchants_df=spark.table("finstream_data_pipeline.bronze.merchants")
merchant_required = transactions_df.filter(
    col("transaction_type").isin(
        "payment",
        "purchase"
    )
)

invalid_transaction_merchants = (
    merchant_required
    .join(
        merchants_df.select("merchant_id"),
        on="merchant_id",
        how="left_anti"
    )
)

print(
    "Transactions with invalid merchant_id:",
    invalid_transaction_merchants.count()
)

Transactions with invalid merchant_id: 0


In [0]:
events_df = spark.table(
    "finstream_data_pipeline.bronze.transaction_events"
)

invalid_events = (
    events_df
    .join(
        transactions_df.select("transaction_id"),
        on="transaction_id",
        how="left_anti"
    )
)

print(
    "Events with invalid transaction_id:",
    invalid_events.count()
)

Events with invalid transaction_id: 0
